In [ ]:
import os

def _pick(*cands):
    for c in cands:
        if os.path.isdir(c):
            return c
    return cands[0]

CODE_DIR = _pick(
    "/kaggle/input/datasets/kavishahaswani/occluded-object-detection-code",
    "/kaggle/input/kavishahaswani/occluded-object-detection-code",
)
MOCA = _pick(
    "/kaggle/input/datasets/ivanomelchenkoim11/moca-dataset/MoCA",
    "/kaggle/input/ivanomelchenkoim11/moca-dataset/MoCA",
)
CAD = _pick(
    "/kaggle/input/datasets/tanushkalokegaonkar/camouflaged-animal-dataset/CamouflagedAnimalDataset",
    "/kaggle/input/tanushkalokegaonkar/camouflaged-animal-dataset/CamouflagedAnimalDataset",
)
YUV = _pick(
    "/kaggle/input/datasets/tanushkalokegaonkar/yuv20k/YUV20K",
    "/kaggle/input/tanushkalokegaonkar/yuv20k/YUV20K",
)
print("CODE:", CODE_DIR)
print("MOCA:", MOCA)
print("CAD:", CAD)
print("YUV:", YUV)
assert os.path.isdir(CODE_DIR), f"code dataset not found under {CODE_DIR}"
print("code files:", sorted(os.listdir(CODE_DIR)))


In [ ]:
import os, sys, subprocess

def run(*args):
    """Run a python script with check=True so any failure STOPS the notebook
    (unlike `!python ...`, which silently continues on error)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True)
    print("OK:", args[0].split("/")[-1], flush=True)

NUM_WORKERS = os.cpu_count() or 1
print("CPU workers for preprocessing:", NUM_WORKERS)


In [ ]:
# 1) Convert MoCA CSV annotations -> GT masks (MUST run before make_split.py,
#    which enumerates sequences from the converted tree).
run(f"{CODE_DIR}/convert_moca_csv_to_masks.py",
    "--moca_root", MOCA,
    "--output_root", "/kaggle/working/moca_converted")
assert os.path.isdir("/kaggle/working/moca_converted")
print("moca_converted subdirs:", sorted(os.listdir("/kaggle/working/moca_converted"))[:10])


In [ ]:
# 2) Build the deterministic sequence-level splits (seed 42).
run(f"{CODE_DIR}/make_split.py",
    "--moca_root", "/kaggle/working/moca_converted",
    "--cad_root", CAD,
    "--yuv_root", YUV,
    "--out", "/kaggle/working/splits")
print(open("/kaggle/working/splits/split_summary.json").read())
for name in ("moca", "cad", "yuv20k"):
    for split in ("train", "val", "test"):
        p = f"/kaggle/working/splits/{name}_{split}.json"
        assert os.path.exists(p), f"missing split file {p}"
print("split files OK:", sorted(os.listdir("/kaggle/working/splits")))


In [ ]:
# 3) Arm 1: RAW (unamplified motion cache + train)
os.makedirs("/kaggle/working/cache_raw", exist_ok=True)
os.makedirs("/kaggle/working/runs_raw", exist_ok=True)
run(f"{CODE_DIR}/preprocess_amplify.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--cache_root", "/kaggle/working/cache_raw",
    "--fps", "24", "--window_size", "40", "--overlap", "20", "--motion_size", "256", "--no_amplify",
    "--num_workers", str(NUM_WORKERS))
run(f"{CODE_DIR}/train.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--adapters", "moca=moca_mask", "cad=cad", "yuv20k=imgs_gt",
    "--cache_root", "/kaggle/working/cache_raw",
    "--split_dir", "/kaggle/working/splits", "--split", "train",
    "--checkpoint_dir", "/kaggle/working/runs_raw", "--arm_name", "raw",
    "--epochs", "10", "--batch_size", "8", "--image_size", "256", "--base_ch", "32", "--seed", "42")
assert os.path.exists("/kaggle/working/runs_raw/best_by_val.pth")
print("ARM 1 (RAW) DONE")


In [ ]:
# 4) Arm 2: AMP_OLD (alpha 15, low 0.2, high 3.0, 4 levels)
os.makedirs("/kaggle/working/cache_amp_old", exist_ok=True)
os.makedirs("/kaggle/working/runs_amp_old", exist_ok=True)
run(f"{CODE_DIR}/preprocess_amplify.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--cache_root", "/kaggle/working/cache_amp_old",
    "--fps", "24", "--window_size", "40", "--overlap", "20", "--motion_size", "256",
    "--alpha", "15.0", "--low_hz", "0.2", "--high_hz", "3.0", "--levels", "4",
    "--num_workers", str(NUM_WORKERS))
run(f"{CODE_DIR}/train.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--adapters", "moca=moca_mask", "cad=cad", "yuv20k=imgs_gt",
    "--cache_root", "/kaggle/working/cache_amp_old",
    "--split_dir", "/kaggle/working/splits", "--split", "train",
    "--checkpoint_dir", "/kaggle/working/runs_amp_old", "--arm_name", "amp_old",
    "--epochs", "10", "--batch_size", "8", "--image_size", "256", "--base_ch", "32", "--seed", "42")
assert os.path.exists("/kaggle/working/runs_amp_old/best_by_val.pth")
print("ARM 2 (AMP_OLD) DONE")


In [ ]:
# 5) Arm 3: AMP_V2 (alpha 5, low 0.5, high 2.5, brightness-only, coarse-only, median-sub, percentile norm)
os.makedirs("/kaggle/working/cache_amp_v2", exist_ok=True)
os.makedirs("/kaggle/working/runs_amp_v2", exist_ok=True)
run(f"{CODE_DIR}/preprocess_amplify.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--cache_root", "/kaggle/working/cache_amp_v2",
    "--fps", "24", "--window_size", "40", "--overlap", "20", "--motion_size", "256",
    "--alpha", "5.0", "--low_hz", "0.5", "--high_hz", "2.5", "--levels", "4",
    "--brightness_only", "--coarse_only", "--median_sub", "--norm_mode", "percentile",
    "--num_workers", str(NUM_WORKERS))
run(f"{CODE_DIR}/train.py",
    "--data_roots", "moca=/kaggle/working/moca_converted", f"cad={CAD}", f"yuv20k={YUV}",
    "--adapters", "moca=moca_mask", "cad=cad", "yuv20k=imgs_gt",
    "--cache_root", "/kaggle/working/cache_amp_v2",
    "--split_dir", "/kaggle/working/splits", "--split", "train",
    "--checkpoint_dir", "/kaggle/working/runs_amp_v2", "--arm_name", "amp_v2",
    "--epochs", "10", "--batch_size", "8", "--image_size", "256", "--base_ch", "32", "--seed", "42")
assert os.path.exists("/kaggle/working/runs_amp_v2/best_by_val.pth")
print("ARM 3 (AMP_V2) DONE")


In [ ]:
# 6) Summary Comparison Table across all 3 arms on Test Split
import json
arms = ["raw", "amp_old", "amp_v2"]
metrics = ["IoU", "Dice", "MAE", "S_measure", "weighted_F"]
header = f"{'Arm':<12} | " + " | ".join(f"{m:>10}" for m in metrics)
print("=" * len(header))
print("3-ARM TEST SPLIT BENCHMARK COMPARISON")
print("=" * len(header))
print(header)
print("-" * len(header))
for arm in arms:
    p = f"/kaggle/working/runs_{arm}/test_report.json"
    if os.path.exists(p):
        rep = json.loads(open(p).read())
        ov = rep.get("overall", {})
        vals = " | ".join(f"{ov.get(m, 0.0):>10.4f}" for m in metrics)
        print(f"{arm:<12} | {vals}")
    else:
        print(f"{arm:<12} | [NOT FOUND]")
print("-" * len(header))
